# Train on `combined_clean.csv`

### Requirements (what this notebook follows)
1. **Inputs for the model:** only the two text columns **`text_a`** and **`text_b`**. No links, outlets, or other CSV fields are loaded.
2. **Target:** **`OVERALL`** (the score to predict). Hugging Face expects a column named **`labels`**, so we set `labels = OVERALL` after loading. The model never sees non-text features—only tokenized **text_a** + **text_b** and the numeric **labels** for training.
3. **Split:** **80% training**, **10% validation**, **10% test** (we hold out 20% of rows, then split that half and half for val vs test). The same split is reused for **both** models below.
4. **Models (two only):** `FacebookAI/xlm-roberta-base` and `google-bert/bert-base-multilingual-cased`. (mDeBERTa and InfoXLM are **not** used.)

### Run order (top to bottom)
1. **Environment** – in Terminal: `cd` to this project and run `./setup_env.sh` (creates `.venv` and installs packages). In the notebook, choose the Python interpreter **`NLI/.venv/bin/python`**.
2. **Imports & settings** – change batch size, epochs, or models if you want.
3. **Load CSV** – read and clean rows.
4. **Split** – build train / val / test.
5. **Functions** – tokenize text pairs, define metrics, define one training routine.
6. **Train loop** – trains **each** model one after another (this takes a long time).
7. **Summary table** – shows test scores side by side.
8. **Save** – writes `runs_multilingual/summary_metrics.csv`.

Trained weights are saved under **`runs_multilingual/`** (one subfolder per model).

In [ ]:
# Step 1) Install (run once per environment)
# Easiest: in Terminal run  ./setup_env.sh  from the NLI folder, then pick kernel .venv/bin/python
# Or uncomment:
# !pip install -r requirements.txt

In [23]:
# Step 2) Imports + knobs you can change (batch size, epochs, model list, etc.)

import os

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

CSV_FILE = "combined_clean.csv"   # overridden to Google Drive path in Step 2b when on Colab
SAVE_FOLDER = "runs_multilingual"  # overridden to Google Drive path in Step 2b when on Colab

MODELS_TO_TRAIN = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-cased",
]

# Maps the 5 discrete OVERALL scores → integer class indices (required by HF classifier)
LABEL_MAP   = {1.0: 0, 2.0: 1, 2.5: 2, 3.0: 3, 4.0: 4}
LABEL_NAMES = ["score_1.0", "score_2.0", "score_2.5", "score_3.0", "score_4.0"]
NUM_CLASSES = len(LABEL_MAP)

RANDOM_SEED = 42
MAX_TOKENS = 256
BATCH_SIZE = 8
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [24]:
# Step 2b) Mount Google Drive (Colab only).
#
# Your Google Drive must have this layout before running:
#   My Drive/NLI/combined_clean.csv   ← input CSV
#
# Trained models + summary_metrics.csv will be saved to:
#   My Drive/NLI/runs_multilingual/   ← persists after session ends

try:
    import google.colab  # noqa: F401
    _on_colab = True
except ImportError:
    _on_colab = False

if _on_colab:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_NLI = "/content/drive/MyDrive/NLI"
    os.makedirs(DRIVE_NLI, exist_ok=True)

    CSV_FILE    = os.path.join(DRIVE_NLI, "combined_clean.csv")
    SAVE_FOLDER = os.path.join(DRIVE_NLI, "runs_multilingual")
    os.makedirs(SAVE_FOLDER, exist_ok=True)

    print("Drive mounted.")
    print(f"  Reading CSV from : {CSV_FILE}")
    print(f"  Saving models to : {SAVE_FOLDER}")

    if not os.path.exists(CSV_FILE):
        print("\n*** FILE NOT FOUND ***")
        print(f"Place  combined_clean.csv  inside your Google Drive folder: {DRIVE_NLI}")
        print("Then re-run this cell.")
    else:
        size_mb = os.path.getsize(CSV_FILE) / 1e6
        print(f"  CSV found ({size_mb:.1f} MB) — ready to load.")
else:
    print("Not on Colab — using local paths (CSV_FILE and SAVE_FOLDER from Step 2).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
  Reading CSV from : /content/drive/MyDrive/NLI/combined_clean.csv
  Saving models to : /content/drive/MyDrive/NLI/runs_multilingual
  CSV found (9.3 MB) — ready to load.


In [25]:
# Step 3) Load the CSV and keep only the two texts + the score we predict

df = pd.read_csv(CSV_FILE)

needed = ["text_a", "text_b", "OVERALL"]
for col in needed:
    if col not in df.columns:
        raise ValueError("csv is missing column: " + col)

df = df[needed].copy()
df["text_a"] = df["text_a"].fillna("").astype(str)
df["text_b"] = df["text_b"].fillna("").astype(str)

df = df.dropna(subset=["OVERALL"])
df = df[df["text_a"].str.strip() != ""]
df = df[df["text_b"].str.strip() != ""]

df["labels"] = df["OVERALL"].map(LABEL_MAP)
unmapped = df["labels"].isna().sum()
if unmapped:
    print(f"WARNING: {unmapped} rows had OVERALL values not in LABEL_MAP and will be dropped.")
    print("Unexpected values:", df.loc[df["labels"].isna(), "OVERALL"].unique())
df = df.dropna(subset=["labels"])
df["labels"] = df["labels"].astype(int)

df = df[["text_a", "text_b", "labels"]].reset_index(drop=True)

print("number of rows after cleaning:", len(df))
print("class distribution:")
for score, idx in sorted(LABEL_MAP.items()):
    count = (df["labels"] == idx).sum()
    print(f"  OVERALL {score} → class {idx} : {count} rows")
df.head()

number of rows after cleaning: 18667
class distribution:
  OVERALL 1.0 → class 0 : 4584 rows
  OVERALL 2.0 → class 1 : 2808 rows
  OVERALL 2.5 → class 2 : 3442 rows
  OVERALL 3.0 → class 3 : 3408 rows
  OVERALL 4.0 → class 4 : 4425 rows


,text_a,text_b,labels
0,Coronavirus: Pandemia La gripe española de 191...,WHO European director urges caution on easing ...,0
1,Konflikte: Israel reagiert mit Luftangriffen a...,Israel strikes in Gaza in response to incendia...,4
2,Corona-Virus: Apple-Mitarbeiter bekommen Care-...,Coronavirus : Foxconn proposerait des primes p...,2
3,隐瞒？试剂不准？看中国专家如何解释,"Governance, technology and citizen behavior in...",0
4,Sturm der Liebe: Neue Folgen bringen neue Dars...,New 2020 Seat Leon teased ahead of launch,0


In [26]:
# Step 4) 80% train, 10% val, 10% test (20% held out, then split 50/50)

n_total = len(df)

train_df, rest_df = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED)
val_df, test_df = train_test_split(rest_df, test_size=0.5, random_state=RANDOM_SEED)

n_train = len(train_df)
n_val = len(val_df)
n_test = len(test_df)

print("train:", n_train, " val:", n_val, " test:", n_test)
print("as fractions of all rows:", round(n_train / n_total, 3), round(n_val / n_total, 3), round(n_test / n_total, 3))

datasets = DatasetDict(
    train=Dataset.from_pandas(train_df, preserve_index=False),
    validation=Dataset.from_pandas(val_df, preserve_index=False),
    test=Dataset.from_pandas(test_df, preserve_index=False),
)

train: 14933  val: 1867  test: 1867
as fractions of all rows: 0.8 0.1 0.1


In [27]:
# Step 5) Helper functions used by Hugging Face Trainer
# - tokenize_pairs: converts raw text to numbers the model reads
# - my_metrics: MSE / MAE / correlation after each eval
# - add_tokens: runs tokenize_pairs on the whole train/val/test set
# - train_model: download model, train, evaluate, save weights


def tokenize_pairs(batch, tokenizer):
    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        truncation=True,
        max_length=MAX_TOKENS,
        padding=False,
    )


def my_metrics(eval_pred):
    logits, true_labels = eval_pred
    predictions = np.argmax(np.asarray(logits), axis=-1)
    true_labels = np.asarray(true_labels, dtype=int)

    # class 0 = OVERALL 1.0 = completely unrelated pairs ("contradict")
    f1_contradict = float(f1_score(true_labels, predictions, labels=[0], average="macro", zero_division=0))
    macro_f1      = float(f1_score(true_labels, predictions, average="macro", zero_division=0))
    acc           = float(accuracy_score(true_labels, predictions))
    kappa         = float(cohen_kappa_score(true_labels, predictions))

    return {
        "macro_f1":     macro_f1,
        "f1_contradict": f1_contradict,
        "accuracy":     acc,
        "cohen_kappa":  kappa,
    }


def add_tokens(tokenizer):
    def batch_fn(batch):
        return tokenize_pairs(batch, tokenizer)

    return datasets.map(batch_fn, batched=True, remove_columns=["text_a", "text_b"])


def train_model(hf_name):
    folder_name = hf_name.replace("/", "_")
    run_dir = os.path.join(SAVE_FOLDER, folder_name)
    os.makedirs(run_dir, exist_ok=True)

    use_cuda = torch.cuda.is_available()
    if use_cuda:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"[GPU] Using: {gpu_name}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("[GPU] No CUDA GPU found — running on CPU (will be slow)")

    tokenizer = AutoTokenizer.from_pretrained(hf_name, use_fast=True)
    tokenized = add_tokens(tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        hf_name,
        num_labels=NUM_CLASSES,      # 5 classes: 1.0 / 2.0 / 2.5 / 3.0 / 4.0
        problem_type="single_label_classification",
    )

    training_args = TrainingArguments(
        output_dir=run_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",       # saves a checkpoint after every epoch → crash-safe
        save_total_limit=2,          # keeps only the 2 most recent checkpoints (saves Drive space)
        load_best_model_at_end=True, # loads the best checkpoint when training finishes
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        logging_steps=50,
        seed=RANDOM_SEED,
        report_to="none",
        bf16=use_cuda,
        fp16=False,
        use_cpu=not use_cuda,
        dataloader_pin_memory=use_cuda,
        optim="adamw_torch",
    )

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        compute_metrics=my_metrics,
    )
    # some library versions want tokenizer=, others want processing_class=
    try:
        trainer = Trainer(tokenizer=tokenizer, **kwargs)
    except TypeError:
        trainer = Trainer(processing_class=tokenizer, **kwargs)

    trainer.train()

    val_scores = trainer.evaluate(tokenized["validation"])
    test_scores = trainer.evaluate(tokenized["test"])

    final_path = os.path.join(run_dir, "final_model")
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)

    return {
        "model": hf_name,
        "val": val_scores,
        "test": test_scores,
        "saved": final_path,
    }

### Step 6) Train every model in the list

This cell **re-trains from scratch** for each name in `MODELS_TO_TRAIN`.  
**Add more checkpoints:** append other Hugging Face model ids to `MODELS_TO_TRAIN` in Step 2 if you need them.

In [28]:
# Step 6 continued) loop: each model gets its own folder under SAVE_FOLDER

results = []

for model_id in MODELS_TO_TRAIN:
    print("\n--- training:", model_id, "---")
    one_result = train_model(model_id)
    results.append(one_result)

    val = one_result["val"]
    tst = one_result["test"]
    print("validation (float metrics only):")
    for k, v in val.items():
        if isinstance(v, float):
            print(" ", k, round(v, 4))
    print("test:")
    for k, v in tst.items():
        if isinstance(v, float):
            print(" ", k, round(v, 4))


--- training: FacebookAI/xlm-roberta-base ---
[GPU] Using: NVIDIA A100-SXM4-40GB  |  VRAM: 42.4 GB


Map:   0%|          | 0/14933 [00:00<?, ? examples/s]

Map:   0%|          | 0/1867 [00:00<?, ? examples/s]

Map:   0%|          | 0/1867 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

validation (float metrics only):
  eval_loss 1.2312
  eval_macro_f1 0.4761
  eval_f1_contradict 0.6162
  eval_accuracy 0.5088
  eval_cohen_kappa 0.3785
  eval_runtime 3.9356
  eval_samples_per_second 474.389
  eval_steps_per_second 59.457
  epoch 10.0
test:
  eval_loss 1.2328
  eval_macro_f1 0.4628
  eval_f1_contradict 0.6092
  eval_accuracy 0.4922
  eval_cohen_kappa 0.3556
  eval_runtime 4.033
  eval_samples_per_second 462.936
  eval_steps_per_second 58.022
  epoch 10.0


In [31]:
# Step 7) One small table comparing all models on the **test** split

summary_rows = []
for r in results:
    test_dict = r["test"]
    row = {
        "model":           r["model"],
        "macro_f1":        test_dict.get("eval_macro_f1"),
        "f1_contradict":   test_dict.get("eval_f1_contradict"),
        "accuracy":        test_dict.get("eval_accuracy"),
        "cohen_kappa":     test_dict.get("eval_cohen_kappa"),
        "saved_to":        r["saved"],
    }
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary

,model,macro_f1,f1_contradict,accuracy,cohen_kappa,saved_to
0,FacebookAI/xlm-roberta-base,0.481345,0.625602,0.507231,0.374914,/content/drive/MyDrive/NLI/runs_multilingual/F...
1,google-bert/bert-base-multilingual-cased,0.462813,0.609207,0.492234,0.355578,/content/drive/MyDrive/NLI/runs_multilingual/g...


In [ ]:
# Step 8) Save that table next to your training folders

out_csv = os.path.join(SAVE_FOLDER, "summary_metrics.csv")
summary.to_csv(out_csv, index=False)
print("saved table to:", out_csv)

In [ ]:
# Colab + GPU (H100) — run this cell to confirm you're on the right runtime
try:
    import google.colab  # noqa: F401

    _on_colab = True
except ImportError:
    _on_colab = False
print("Google Colab runtime:", "yes" if _on_colab else "no (local or other)")

import torch

def _gpu_family(name: str) -> str:
    n = (name or "").lower()
    for tag, label in [
        ("h100", "H100"),
        ("h200", "H200"),
        ("a100", "A100"),
        ("a800", "A800"),
        ("l4", "L4"),
        ("l40", "L40"),
        ("a40", "A40"),
        ("t4", "T4"),
        ("v100", "V100"),
    ]:
        if tag in n:
            return label
    return "other / unknown"

print("PyTorch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    print("GPU 0:", dev_name)
    fam = _gpu_family(dev_name)
    print("Detected family:", fam, "(from name string; not performance)")
    want_h100 = fam == "H100"
    print("Is H100:", want_h100)
    if not want_h100:
        print(
            "Note: 'Is H100: False' means this VM has a different chip (e.g. A100). "
            "In Colab: Runtime → Change runtime type → pick an H100 shape if your subscription shows one."
        )
else:
    print("No CUDA GPU visible to PyTorch — not on a GPU runtime or driver issue.")

import subprocess

try:
    smi = subprocess.check_output(["nvidia-smi", "-L"], text=True, timeout=10)
    print("nvidia-smi -L:\n", smi.strip())
except Exception as e:
    print("nvidia-smi (optional):", e)